## Importing Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
!pip install mlflow
!pip install mlflow dagshub

from sklearn.model_selection import train_test_split

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Regression Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## DagsHub MLflow Setup

In [ ]:
import dagshub

dagshub.init(
    repo_owner="kanishka",
    repo_name="MLflow",
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3aebf7ce-9f88-4abb-9879-a92966fb87eb&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2a9a8f12abed4e00192ab36e4e168f29b1b93599210195b979ac9c9daee4b292




Accessing as kanishka

Initialized MLflow to track repo "kanishka/MLflow"

Repository kanishka/MLflow initialized!

In [ ]:
mlflow.set_experiment(
    "Housing Price Prediction PBLM 1"
)

2026/08/07 06:19:57 INFO mlflow.tracking.fluent: Experiment with name 'Housing Price Prediction PBLM 1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3a524767beaa4840ab001a09a3581784', creation_time=1786083597404, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786083597404, lifecycle_stage='active', name='Housing Price Prediction PBLM 1', tags={}, trace_location=None, workspace='default'>

## Data Loading and Processing

In [ ]:
import pandas as pd

data = pd.read_csv(r"/content/HousingData.csv")

# Remove missing values
data = data.dropna()

# Features and Target
X = data.drop("MEDV", axis=1)
y = data["MEDV"]

print("Shape:", X.shape)
print("Target Column: MEDV")
print("Target Statistics:")
print(y.describe())

Shape: (394, 13)
Target Column: MEDV
Target Statistics:
count    394.000000
mean      22.359645
std        9.142979
min        5.000000
25%       16.800000
50%       21.050000
75%       25.000000
max       50.000000
Name: MEDV, dtype: float64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Build Models

In [ ]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [ ]:
reports = []
trained_models = []

for model_name, model, X_tr, y_tr in models:

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    report = {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

Linear Regression
MAE : 3.4558
MSE : 28.8708
RMSE: 5.3732
R²  : 0.6905
Random Forest
MAE : 2.5791
MSE : 20.9917
RMSE: 4.5817
R²  : 0.7750
XGBoost
MAE : 2.5938
MSE : 20.6410
RMSE: 4.5432
R²  : 0.7787


## Log All Experiments to DagsHub

In [ ]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Parameters
        mlflow.log_param("Model", model_name)
        mlflow.log_params(model.get_params())

        # Regression Metrics
        mlflow.log_metric("MAE", report["MAE"])
        mlflow.log_metric("MSE", report["MSE"])
        mlflow.log_metric("RMSE", report["RMSE"])
        mlflow.log_metric("R2", report["R2"])

        # Log Model
        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged successfully!")

2026/08/07 06:23:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0/runs/f6f061ba3dbe4cd091b2bc8082a77bef
🧪 View experiment at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0


2026/08/07 06:24:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0/runs/771aa98ab3da4d938cb51b7f23dd2167
🧪 View experiment at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0


2026/08/07 06:24:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0/runs/932f889f70ef4f7f9713fc35222eaf8c
🧪 View experiment at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0
Experiments logged successfully!


## Best Model and Reg to DH

In [ ]:
best_index = np.argmax(
    [r["R2"] for r in reports]
)

best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print("Best Model:", best_model_name)
print("R2 Score :", best_report["R2"])
print("MAE      :", best_report["MAE"])
print("MSE      :", best_report["MSE"])
print("RMSE     :", best_report["RMSE"])

Best Model: XGBoost
R2 Score : 0.7787370768912256
MAE      : 2.5937674995230022
MSE      : 20.641015135430465
RMSE     : 4.543238397380272


In [ ]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    # Parameters
    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_param(
        "Selection_Metric",
        "R2 Score"
    )

    # Metrics
    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    # Log model
    if "XGBoost" in best_model_name:

        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    else:

        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID    :", run_id)
print("Model URI :", model_uri)
print("Model Name:", "Housing_Price_Best_Model")

2026/08/07 06:25:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Housing_Price_Best_Model'.
2026/08/07 06:26:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Housing_Price_Best_Model, version 1
Created version '1' of model 'Housing_Price_Best_Model'.


🏃 View run Champion_XGBoost at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0/runs/22e1a685397246e5837f357152ad38ba
🧪 View experiment at: https://dagshub.com/kanishka/MLflow.mlflow/#/experiments/0
Run ID    : 22e1a685397246e5837f357152ad38ba
Model URI : models:/m-7cf6ed361ed542b38e7f3c479c33396c
Model Name: Housing_Price_Best_Model
